# 10f Stand-alone Automated Concept Refinement Cycle

This combines the inspectable stages in 10a–10d: prepare the pair-year corpus and split, create a mutable working catalog from the seed reference or resume the current working catalog, classify with the same Jev cache, build yearly signals, and run development forecasts. It repeats the proposal/classify/evaluate stages as a development-only refinement loop. This notebook can run from a clean checkout without first running 10a or 10b.

Edit the parameter block in the setup cell for pathogen limit, split, models, classification, forecasts, and refinement size. Defaults match the current 10a configuration and the configured project models. The default search uses three rounds, up to three Pareto-front parents per round, and three children per parent.

Jev labels are cached per abstract and concept definition, so rerunning or adding children reuses unchanged labels. Proposals, round summaries, forecasts, catalog snapshots, and Pareto tables are saved under `outputs/concept_search/automated_refinement/`. Confirmation remains separate in notebook 10e after the catalog is frozen.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import (
    build_si_concept_paths, corpus_preflight, load_concept_corpus,
)
from amr.screening.system1 import OpenRouterJevClient
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, concept_definition_hash, JEV_CLASSIFICATION_PROMPT_VERSION, pareto_concept_frontier,
    classify_concept_abstracts_jev, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)

# -----------------------------------------------------------------------------
# Experiment parameters: edit these values before running the notebook.
# Corpus and split (same rules as notebook 10a; all antibiotics are retained).
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)
SI_PATH = PROJECT_ROOT / "outputs" / "parquets" / "yearly_si.parquet"
PATHOGEN_PRIORITY_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_priority", PROJECT_ROOT)
MIN_PATHOGENIC_INDEX = float(cfg(PIPELINE_CONFIG, "collection.min_pathogenic_index", 0))
NUM_PATHOGENS = 10              # None uses all eligible pathogens; integer keeps all antibiotics for top-ranked pathogens.
HELDOUT_FRACTION = 0.20        # Pair-level confirmation fraction, excluded from proposal and selection.
SPLIT_SEED = 42
RESET_CATALOG_TO_SEEDS = True # True starts from the nine seed concepts; otherwise resume the saved catalog if present.

# Jev classification and LLM proposal parameters.
JEV_MODEL = str(cfg(PIPELINE_CONFIG, "system1.model", "typesafe/jev-1.13"))
MAX_WORKERS = int(cfg(PIPELINE_CONFIG, "system1.max_workers", 128))
MAX_CLASSIFICATION_CALLS = None # None completes all missing labels; a cap pauses safely and requires rerunning to resume.
LLM_MODEL = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))
LLM_TEMPERATURE = 0
LLM_MAX_TOKENS = 4000
MAX_EXAMPLES_PER_PARENT = 30
PROPOSAL_EXAMPLE_SEED = 42
POSITIVE_EXAMPLE_THRESHOLD = 0.5

# Forecast/evaluation parameters. Training is expanding-window through each origin.
FIT_START_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_start_year", 2000))
FORECAST_ORIGIN_END_YEAR = int(cfg(PIPELINE_CONFIG, "si.fit_end_year", 2023))
LAG_LENGTH = 3
FORECAST_HORIZONS = (1, 2, 3)
MIN_PRIOR_OBSERVATIONS = 10
FORECAST_PROGRESS_EVERY = 10
FRONTIER_DISPLAY_ROWS = 30

# Automated search parameters.
N_ITER = 3
K_PARENTS = 3
NUM_CHILDREN = 3
RUN_AUTOMATED_CYCLE = True
# -----------------------------------------------------------------------------


## Prepare corpus, split, and starting catalog

This stage duplicates notebook 10a’s preparation so 10f is runnable on its own: it joins screened PMID membership, cached abstracts, and observed SI; applies the pathogen limit before pair splitting; writes the corpus artifacts; and prints coverage/request-volume preflight before any Jev or LLM calls. The split is deterministic by pathogen–antibiotic pair.

The confirmation partition is kept out of proposal examples and model selection. Shared PMIDs are excluded from proposal examples even when the article appears in both pair partitions. `RESET_CATALOG_TO_SEEDS=True` copies the nine seed concepts into a fresh working catalog for this run; the hash-named seed snapshot remains as the immutable reference. Each iteration extends the working catalog and writes a versioned snapshot. Set it to `False` to load the current working catalog and add another `N_ITER` rounds on a later run. Classification scores remain cached by PMID and concept definition.


The setup cell exposes input paths, pathogen cap, split seed/fraction, catalog reset behavior, Jev and LLM models, worker/call limits, prompt example threshold and sample size, fit years, lag length, forecast horizons, minimum prior observations, progress interval, and refinement counts. Change those values there; the notebook records the corpus/catalog and forecast settings in its run directory name.


If MAX_CLASSIFICATION_CALLS is capped, 10f stops before forecasting when labels remain missing; rerun to resume without treating missing labels as zero prevalence.


In [ ]:
if not 0 < HELDOUT_FRACTION < 1:
    raise ValueError("HELDOUT_FRACTION must be strictly between 0 and 1.")
if NUM_PATHOGENS is not None and (not isinstance(NUM_PATHOGENS, int) or NUM_PATHOGENS < 1):
    raise ValueError("NUM_PATHOGENS must be a positive integer or None.")
integer_parameters = {
    "N_ITER": N_ITER, "K_PARENTS": K_PARENTS, "NUM_CHILDREN": NUM_CHILDREN,
    "LAG_LENGTH": LAG_LENGTH, "MIN_PRIOR_OBSERVATIONS": MIN_PRIOR_OBSERVATIONS,
    "MAX_EXAMPLES_PER_PARENT": MAX_EXAMPLES_PER_PARENT, "MAX_WORKERS": MAX_WORKERS,
}
if any(not isinstance(value, int) or isinstance(value, bool) for value in integer_parameters.values()):
    raise ValueError(f"These parameters must be integers: {integer_parameters}")
if any(value < 1 for value in integer_parameters.values()):
    raise ValueError(f"These parameters must be positive: {integer_parameters}")
if not FORECAST_HORIZONS or any(not isinstance(h, int) or isinstance(h, bool) or h < 1 for h in FORECAST_HORIZONS):
    raise ValueError("FORECAST_HORIZONS must contain positive integers.")
if len(set(FORECAST_HORIZONS)) != len(FORECAST_HORIZONS):
    raise ValueError("FORECAST_HORIZONS must not contain duplicates.")
if FIT_START_YEAR > FORECAST_ORIGIN_END_YEAR:
    raise ValueError("FIT_START_YEAR must be no later than FORECAST_ORIGIN_END_YEAR.")
if not 0 <= POSITIVE_EXAMPLE_THRESHOLD <= 1:
    raise ValueError("POSITIVE_EXAMPLE_THRESHOLD must be between 0 and 1.")
if MAX_CLASSIFICATION_CALLS is not None and (
    not isinstance(MAX_CLASSIFICATION_CALLS, int) or isinstance(MAX_CLASSIFICATION_CALLS, bool)
    or MAX_CLASSIFICATION_CALLS < 0
):
    raise ValueError("MAX_CLASSIFICATION_CALLS must be a nonnegative integer or None.")

corpus, pair_split = load_concept_corpus(
    screened_path=YEARLY_SCREENED_PATH,
    abstracts_path=ABSTRACT_CACHE_PATH,
    si_path=SI_PATH,
    heldout_fraction=HELDOUT_FRACTION,
    split_seed=SPLIT_SEED,
    pathogen_priority_path=PATHOGEN_PRIORITY_PATH,
    num_pathogens=NUM_PATHOGENS,
    min_pathogenic_index=MIN_PATHOGENIC_INDEX,
)
CONCEPT_PATHS["directory"].mkdir(parents=True, exist_ok=True)
abstract_table = corpus[["pmid", "title", "abstract", "abstract_available"]].drop_duplicates("pmid")
corpus.drop(columns=["title", "abstract"]).to_parquet(CONCEPT_PATHS["corpus"], index=False)
abstract_table.to_parquet(CONCEPT_PATHS["abstracts"], index=False)
pair_split.to_csv(CONCEPT_PATHS["pair_split"], index=False)
preflight = corpus_preflight(corpus)

CONCEPT_PATHS["catalog_versions"].mkdir(parents=True, exist_ok=True)
reference_catalog = [dict(concept) for concept in SEED_CONCEPTS]
REFERENCE_CATALOG_VERSION = concept_catalog_hash(reference_catalog)
reference_path = CONCEPT_PATHS["catalog_versions"] / f"concept_catalog_{REFERENCE_CATALOG_VERSION}.json"
if not reference_path.exists():
    reference_path.write_text(json.dumps(reference_catalog, indent=2, ensure_ascii=False))

if RESET_CATALOG_TO_SEEDS or not CONCEPT_PATHS["catalog"].exists():
    catalog = [dict(concept) for concept in reference_catalog]
    CONCEPT_PATHS["catalog"].write_text(json.dumps(catalog, indent=2, ensure_ascii=False))
else:
    catalog = json.loads(CONCEPT_PATHS["catalog"].read_text())

# Keep an immutable snapshot of this run's starting definition; `catalog` below
# is the mutable working copy that receives children in each iteration.
STARTING_CATALOG_VERSION = concept_catalog_hash(catalog)
starting_catalog_path = CONCEPT_PATHS["catalog_versions"] / f"concept_catalog_{STARTING_CATALOG_VERSION}.json"
if not starting_catalog_path.exists():
    starting_catalog_path.write_text(json.dumps(catalog, indent=2, ensure_ascii=False))

si_df = pd.read_parquet(SI_PATH)
split = pair_split.copy()
split[["pathogen", "antibiotic"]] = split[["pathogen", "antibiotic"]].astype(str)
development_pairs = set(map(tuple, split.loc[split["split"].eq("development"), ["pathogen", "antibiotic"]].to_numpy()))
development_pmids = set(corpus.loc[corpus["split"].eq("development"), "pmid"].astype(str))
confirmation_pmids = set(corpus.loc[corpus["split"].eq("confirmation"), "pmid"].astype(str))
development_only_pmids = development_pmids - confirmation_pmids
abstract_table["pmid"] = abstract_table["pmid"].astype(str)
corpus["pmid"] = corpus["pmid"].astype(str)
abstracts = abstract_table.loc[
    abstract_table["abstract_available"].astype(bool), ["pmid", "title", "abstract"]
].drop_duplicates("pmid")
abstracts = abstracts.loc[abstracts["abstract"].fillna("").astype(str).str.strip().ne("")]
if abstracts.empty:
    raise RuntimeError("The selected corpus has no nonblank abstracts to classify; concept signals cannot be evaluated.")

# Estimate actual pending Jev requests from the concept-definition cache. Jev
# receives one request per abstract with one or more missing concept questions.
abstracts["pmid"] = abstracts["pmid"].map(
    lambda value: str(value).strip()[:-2]
    if str(value).strip().endswith(".0") and str(value).strip()[:-2].isdigit()
    else str(value).strip()
)
definition_versions = {
    str(concept["concept_id"]): concept_definition_hash(concept)
    for concept in catalog
}
cached_by_concept = {concept_id: set() for concept_id in definition_versions}
if CONCEPT_PATHS["scores"].exists():
    cached_scores = pd.read_parquet(CONCEPT_PATHS["scores"])
    required_cache_columns = {
        "pmid", "concept_id", "concept_version", "model_name", "prompt_version", "status",
    }
    if required_cache_columns.issubset(cached_scores.columns):
        reusable = cached_scores.loc[
            cached_scores["model_name"].astype(str).eq(JEV_MODEL)
            & cached_scores["prompt_version"].astype(str).eq(JEV_CLASSIFICATION_PROMPT_VERSION)
            & cached_scores["status"].eq("ok")
        ]
        for concept_id, concept_version in definition_versions.items():
            concept_rows = reusable.loc[
                reusable["concept_id"].astype(str).eq(concept_id)
                & reusable["concept_version"].astype(str).eq(concept_version), "pmid"
            ]
            cached_by_concept[concept_id] = set(concept_rows.astype(str))
pending_jev_pmids = {
    pmid for pmid in abstracts["pmid"].astype(str)
    if any(pmid not in cached_by_concept[concept_id] for concept_id in definition_versions)
}
del cached_by_concept
CATALOG_VERSION = concept_catalog_hash(catalog)
CORPUS_KEY = hashlib.sha256(
    ("|".join(sorted(corpus["pmid"].astype(str).unique())) + "|" +
     "|".join(f"{p}\t{a}\t{s}" for p, a, s in sorted(map(tuple, split[["pathogen", "antibiotic", "split"]].to_numpy())))).encode("utf-8")
).hexdigest()[:12]
horizon_tag = "-".join(map(str, FORECAST_HORIZONS))
run_configuration = {
    "corpus_key": CORPUS_KEY,
    "catalog_version": CATALOG_VERSION,
    "jev_model": JEV_MODEL,
    "llm_model": LLM_MODEL,
    "llm_temperature": LLM_TEMPERATURE,
    "llm_max_tokens": LLM_MAX_TOKENS,
    "max_examples_per_parent": MAX_EXAMPLES_PER_PARENT,
    "positive_example_threshold": POSITIVE_EXAMPLE_THRESHOLD,
    "proposal_example_seed": PROPOSAL_EXAMPLE_SEED,
    "fit_start_year": FIT_START_YEAR,
    "forecast_origin_end_year": FORECAST_ORIGIN_END_YEAR,
    "lag_length": LAG_LENGTH,
    "forecast_horizons": list(FORECAST_HORIZONS),
    "min_prior_observations": MIN_PRIOR_OBSERVATIONS,
    "n_iter": N_ITER,
    "k_parents": K_PARENTS,
    "num_children": NUM_CHILDREN,
}
RUN_KEY = hashlib.sha256(json.dumps(run_configuration, sort_keys=True).encode("utf-8")).hexdigest()[:8]
RUN_DIR = CONCEPT_PATHS["automated_refinement"] / (
    f"corpus_{CORPUS_KEY}_catalog_{CATALOG_VERSION}_n{N_ITER}_k{K_PARENTS}_c{NUM_CHILDREN}"
    f"_fit{FIT_START_YEAR}-{FORECAST_ORIGIN_END_YEAR}_l{LAG_LENGTH}_h{horizon_tag}_{RUN_KEY}"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert corpus[["pathogen", "antibiotic", "year", "pmid"]].duplicated().sum() == 0
assert set(pair_split["split"]) == {"development", "confirmation"}
print({
    "selected_pathogens": int(corpus["pathogen"].nunique()),
    "pathogen_limit": NUM_PATHOGENS,
    "all_antibiotics_retained": True,
    "pair_year_pmid_associations": preflight["pair_year_pmid_associations"],
    "unique_pmids": preflight["unique_pmids"],
    "unique_nonblank_abstracts_to_classify": preflight["unique_abstracts_to_classify"],
    "abstract_coverage": preflight["abstract_coverage"],
    "full_corpus_jev_request_upper_bound_if_uncached": preflight["unique_abstracts_to_classify"],
    "estimated_pending_jev_requests_after_cache": len(pending_jev_pmids),
    "abstracts_with_all_current_concept_labels_cached": len(abstracts) - len(pending_jev_pmids),
    "development_pairs": len(development_pairs),
    "confirmation_pairs": int(split["split"].eq("confirmation").sum()),
    "starting_catalog_version": CATALOG_VERSION,
    "starting_catalog_reference": str(starting_catalog_path),
    "seed_reference_version": REFERENCE_CATALOG_VERSION,
    "starting_concepts": len(catalog),
    "run_directory": str(RUN_DIR),
})


## Run the automated development loop

The first evaluation scores the starting catalog. In each round, the notebook computes a Pareto frontier from equal-weight gains across `FORECAST_HORIZONS`. A concept is on the frontier when no other evaluated concept has both higher generality and higher specialist strength. From the unexpanded frontier it selects up to `K_PARENTS`, ordered by average gain and then specialist strength.

For each selected parent, the LLM receives up to `MAX_EXAMPLES_PER_PARENT` positive, development-exclusive abstract examples and the parent definition. It is asked for `NUM_CHILDREN` child concepts. The notebook saves each proposal, adds valid nonduplicate children to the catalog, asks Jev only for missing child labels, and evaluates those children on development pairs. New children enter the Pareto pool for the next round. Each round also estimates conditional contribution by comparing the selected population with and without each selected concept. For every expanded parent, nested estimates compare the full selected population against the same Pareto survivor population with that parent replaced by its children; single-child replacement estimates are saved too. These comparisons hold the remaining population fixed and never add children on top of their parent. If fewer than `K_PARENTS` unexpanded frontier concepts remain, the cycle stops early.

The procedure is automated, so inspect its proposals and selection summaries before interpreting them. The 90th-percentile objective protects specialist concepts during selection; the final parent ranking within that frontier uses average improvement, as requested.

For every accepted child, the iteration output prints its name, definition, parent, rationale, inclusion/exclusion criteria, evidence, competing concept, distinction, expected timing, hypothesis, and rejection conditions. The full response is also retained in that parent’s saved proposal JSON.


Progress messages are printed for the initial evaluation, each iteration, each selected parent, proposal/cache loading, classification, and evaluation. The selected Pareto parents are listed with generality and 90th-percentile specialist strength; accepted children print their full refinement details. Jev classification and rolling forecasts also report progress and ETA.


In [ ]:
if not RUN_AUTOMATED_CYCLE:
    print("Automated refinement is disabled. Set RUN_AUTOMATED_CYCLE=True to run the cycle.")
else:
    from amr.llm import OpenAIConfig, OpenAILanguageModel

    def progress(message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] {message}", flush=True)

    llm = OpenAILanguageModel(OpenAIConfig(model=LLM_MODEL, temperature=LLM_TEMPERATURE, max_tokens=LLM_MAX_TOKENS))
    jev = OpenRouterJevClient(model=JEV_MODEL)
    fit_start = FIT_START_YEAR
    fit_end = FORECAST_ORIGIN_END_YEAR
    expanded_parent_ids = set()
    evaluated_concept_ids = [str(concept["concept_id"]) for concept in catalog]
    all_population_summaries = []
    all_pair_summaries = []
    round_log = []

    def save_catalog_snapshot(current_catalog):
        version = concept_catalog_hash(current_catalog)
        CONCEPT_PATHS["catalog_versions"].mkdir(parents=True, exist_ok=True)
        snapshot = CONCEPT_PATHS["catalog_versions"] / f"concept_catalog_{version}.json"
        snapshot.write_text(json.dumps(current_catalog, indent=2, ensure_ascii=False))
        CONCEPT_PATHS["catalog"].write_text(json.dumps(current_catalog, indent=2, ensure_ascii=False))
        return version

    def classify_and_signal(current_catalog, stage_label):
        version = concept_catalog_hash(current_catalog)
        progress(
            f"{stage_label}: checking Jev cache/classification for "
            f"{len(abstracts):,} nonblank abstracts × {len(current_catalog)} concepts."
        )
        scores = classify_concept_abstracts_jev(
            abstracts, current_catalog, jev_client=jev, model_name=JEV_MODEL,
            cache_path=CONCEPT_PATHS["scores"], max_calls=MAX_CLASSIFICATION_CALLS, max_workers=MAX_WORKERS,
        )
        # Never turn unclassified abstracts into implicit zero-prevalence evidence.
        nonblank_pmids = set(
            abstracts.loc[abstracts["abstract"].fillna("").astype(str).str.strip().ne(""), "pmid"].astype(str)
        )
        expected_labels = {
            (pmid, str(concept["concept_id"]))
            for pmid in nonblank_pmids for concept in current_catalog
        }
        successful = scores.loc[
            scores["catalog_version"].astype(str).eq(version)
            & scores["model_name"].astype(str).eq(JEV_MODEL)
            & scores["status"].eq("ok"), ["pmid", "concept_id"]
        ]
        scored_labels = set(zip(successful["pmid"].astype(str), successful["concept_id"].astype(str)))
        missing_labels = expected_labels - scored_labels
        if missing_labels:
            raise RuntimeError(
                f"Classification is incomplete for {len(missing_labels):,} abstract/concept labels "
                f"({len(nonblank_pmids):,} nonblank abstracts × {len(current_catalog)} concepts expected). "
                "No forecasts were run with partial labels. Rerun this cell to resume from the cache, "
                "or set MAX_CLASSIFICATION_CALLS=None to finish all pending labels."
            )
        progress(f"{stage_label}: all required Jev labels are available; building yearly concept signals.")
        signals = build_yearly_concept_signals(
            corpus, scores, catalog_version=version, model_name=JEV_MODEL,
        )
        signals.to_parquet(CONCEPT_PATHS["signals"], index=False)
        progress(f"{stage_label}: signals ready ({len(signals):,} pair-year/concept rows).")
        return scores, signals, version

    def evaluate_concepts(concept_ids, label, signals, survivor_population=None, children_by_parent=None):
        concept_ids = list(dict.fromkeys(map(str, concept_ids)))
        survivor_population = list(dict.fromkeys(map(str, survivor_population or [])))
        children_by_parent = children_by_parent or {}
        model_sets = {f"concept:{cid}": [cid] for cid in concept_ids}
        comparisons = {f"concept:{cid}": "baseline" for cid in concept_ids}

        # Conditional contribution: compare the selected population with and without
        # each member. This asks whether a concept adds information beyond its peers.
        if survivor_population:
            for concept_id in survivor_population:
                full_name = f"conditional_with:{label}:{concept_id}"
                without_name = f"conditional_without:{label}:{concept_id}"
                model_sets[full_name] = list(survivor_population)
                model_sets[without_name] = [cid for cid in survivor_population if cid != concept_id]
                comparisons[full_name] = without_name

            # Nested hierarchy: hold the other selected concepts fixed while replacing
            # each parent with its child set; also report each child replacement alone.
            for parent_id, child_ids in children_by_parent.items():
                child_ids = list(dict.fromkeys(map(str, child_ids)))
                background = [cid for cid in survivor_population if cid != parent_id]
                parent_name = f"nested_parent:{label}:{parent_id}"
                children_name = f"nested_children:{label}:{parent_id}"
                model_sets[parent_name] = list(survivor_population)
                model_sets[children_name] = [*background, *child_ids]
                comparisons[children_name] = parent_name
                for child_id in child_ids:
                    single_name = f"nested_single_child:{label}:{parent_id}:{child_id}"
                    model_sets[single_name] = [*background, child_id]
                    comparisons[single_name] = parent_name

        model_concepts = list(dict.fromkeys(
            concept_ids
            + survivor_population
            + [child for children in children_by_parent.values() for child in children]
        ))
        forecasts = rolling_origin_concept_forecasts(
            si_df, signals, concept_ids=model_concepts, model_sets=model_sets,
            development_pairs=development_pairs, fit_start_year=fit_start, fit_end_year=fit_end,
            lag_length=LAG_LENGTH, horizons=FORECAST_HORIZONS, min_prior_observations=MIN_PRIOR_OBSERVATIONS,
            progress_every=FORECAST_PROGRESS_EVERY,
        )
        if forecasts.empty:
            return forecasts, pd.DataFrame(), pd.DataFrame()
        forecasts = forecasts.merge(split, on=["pathogen", "antibiotic"], how="left", validate="many_to_one")
        forecasts = forecasts.loc[forecasts["split"].eq("development")].drop(columns="split")
        pair_summary, population_summary = summarise_incremental_forecast_gains(forecasts, comparisons)
        safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", label)
        forecasts.to_parquet(RUN_DIR / f"{safe_label}_forecasts.parquet", index=False)
        pair_summary.to_parquet(RUN_DIR / f"{safe_label}_pair_summary.parquet", index=False)
        population_summary.to_parquet(RUN_DIR / f"{safe_label}_population_summary.parquet", index=False)
        return forecasts, pair_summary, population_summary

    # A. Classify and evaluate the starting catalog; concept-specific cache hits are reused.
    save_catalog_snapshot(catalog)
    progress(f"Starting catalog evaluation: {len(evaluated_concept_ids)} concepts; catalog {CATALOG_VERSION}.")
    scores, signals, current_catalog_version = classify_and_signal(catalog, "Initial catalog")
    progress("Initial catalog: beginning rolling-origin forecast evaluation.")
    _, initial_pair_summary, initial_population_summary = evaluate_concepts(
        evaluated_concept_ids, "initial_catalog", signals,
    )
    if initial_population_summary.empty:
        raise RuntimeError("No development forecast summaries were produced; inspect SI coverage and fit years.")
    all_pair_summaries.append(initial_pair_summary)
    all_population_summaries.append(initial_population_summary)
    progress(
        f"Initial catalog evaluation complete: {len(initial_pair_summary):,} pair/horizon rows; "
        f"{len(initial_population_summary):,} population summary rows."
    )

    # B. Expand the strongest unexpanded concepts from the development Pareto frontier.
    for iteration in range(1, N_ITER + 1):
        progress(f"Iteration {iteration}/{N_ITER}: combining current concept results and selecting parents.")
        combined_population = pd.concat(all_population_summaries, ignore_index=True)
        frontier = pareto_concept_frontier(
            combined_population, horizon=0, excluded_concepts=expanded_parent_ids,
        )
        frontier_path = RUN_DIR / f"iteration_{iteration:02d}_pareto_frontier.csv"
        frontier.to_csv(frontier_path, index=False)
        eligible_frontier = frontier.loc[frontier["on_pareto_front"] & frontier["eligible_for_expansion"]].sort_values(
            ["generality", "specialist_strength"], ascending=[False, False],
        )
        survivor_ids = frontier.loc[frontier["on_pareto_front"], "concept_id"].astype(str).tolist()
        selected_parents = eligible_frontier.head(K_PARENTS)
        parent_ids = selected_parents["concept_id"].astype(str).tolist()
        progress(
            f"Iteration {iteration}/{N_ITER}: {int(frontier['on_pareto_front'].sum())} concepts on the Pareto frontier; "
            f"{len(eligible_frontier)} are eligible for expansion."
        )
        if not parent_ids:
            progress(f"Iteration {iteration}/{N_ITER}: no unexpanded Pareto concepts remain; stopping early.")
            break
        progress(f"Iteration {iteration}/{N_ITER}: selected {len(parent_ids)} parent(s) for expansion:")
        frontier_names = {str(concept["concept_id"]): str(concept.get("name", concept["concept_id"])) for concept in catalog}
        for rank, (_, row) in enumerate(selected_parents.iterrows(), start=1):
            concept_id = str(row["concept_id"])
            progress(
                f"  Parent {rank}/{len(parent_ids)}: {frontier_names.get(concept_id, concept_id)} [{concept_id}] "
                f"| average gain={row['generality']:.6g} "
                f"| q90 specialist gain={row['specialist_strength']:.6g}"
            )

        current_by_id = {str(concept["concept_id"]): concept for concept in catalog}
        existing_ids = set(current_by_id)
        existing_definitions = {
            re.sub(r"\s+", " ", str(concept.get("definition", "")).strip().lower())
            for concept in catalog
        }
        new_children = []
        parent_results = []
        for parent_rank, parent_id in enumerate(parent_ids, start=1):
            parent = current_by_id[parent_id]
            progress(
                f"Iteration {iteration}/{N_ITER}: parent {parent_rank}/{len(parent_ids)} "
                f"{parent.get('name', parent_id)} [{parent_id}] — preparing development-only examples."
            )
            parent_scores = scores.loc[
                scores["catalog_version"].astype(str).eq(current_catalog_version)
                & scores["model_name"].astype(str).eq(JEV_MODEL)
                & scores["status"].eq("ok")
                & scores["concept_id"].astype(str).eq(parent_id)
                & scores["pmid"].astype(str).isin(development_only_pmids)
                & pd.to_numeric(scores["probability"], errors="coerce").ge(POSITIVE_EXAMPLE_THRESHOLD)
            ].drop_duplicates("pmid", keep="last")
            examples_df = (
                parent_scores[["pmid"]].merge(abstract_table, on="pmid", how="inner", validate="one_to_one")
                .loc[lambda frame: frame["abstract_available"].astype(bool)]
                .sample(frac=1, random_state=PROPOSAL_EXAMPLE_SEED + iteration - 1)
                .head(MAX_EXAMPLES_PER_PARENT)
            )
            if examples_df.empty:
                progress(f"Iteration {iteration}: {parent_id} has no positive development-only examples; skipped.")
                expanded_parent_ids.add(parent_id)
                parent_results.append({"parent_id": parent_id, "children": [], "status": "no_examples"})
                continue
            proposal_path = RUN_DIR / f"iteration_{iteration:02d}_proposal_{parent_id}.json"
            if proposal_path.exists():
                progress(f"Iteration {iteration}: loading saved proposal for {parent_id} from {proposal_path.name}.")
                proposal = json.loads(proposal_path.read_text())
            else:
                progress(f"Iteration {iteration}: requesting up to {NUM_CHILDREN} child hypotheses for {parent_id}.")
                proposal = propose_concept_children(
                    parent, examples_df[["title", "abstract"]].to_dict("records"),
                    llm=llm, model_name=LLM_MODEL, num_children=NUM_CHILDREN,
                )
                proposal_path.write_text(json.dumps(proposal, indent=2, ensure_ascii=False))
                progress(f"Iteration {iteration}: proposal saved for {parent_id}.")
            proposed_count = len(proposal.get("children", [])) if isinstance(proposal.get("children", []), list) else 0
            progress(f"Iteration {iteration}: proposal for {parent_id} contains {proposed_count} child candidate(s).")
            accepted_ids = []
            for child in proposal.get("children", [])[:NUM_CHILDREN]:
                name = str(child.get("name", "")).strip()
                definition = str(child.get("definition", "")).strip()
                if not name or not definition:
                    continue
                normalized_definition = re.sub(r"\s+", " ", definition.lower())
                if normalized_definition in existing_definitions:
                    continue
                slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_") or "child"
                child_id = f"{parent_id}_{slug}"
                suffix = 2
                while child_id in existing_ids:
                    child_id = f"{parent_id}_{slug}_{suffix}"
                    suffix += 1
                existing_ids.add(child_id)
                existing_definitions.add(normalized_definition)
                review_fields = (
                    "scientific_rationale",
                    "inclusion_criteria",
                    "exclusion_criteria",
                    "representative_evidence",
                    "closest_competing_concept",
                    "difference",
                    "expected_temporal_relationship",
                    "testable_hypothesis",
                    "rejection_conditions",
                )
                child_record = {
                    "concept_id": child_id,
                    "name": name,
                    "parent_id": parent_id,
                    "definition": definition,
                    "inclusion_criteria": child.get("inclusion_criteria", []),
                    "exclusion_criteria": child.get("exclusion_criteria", []),
                    "version": iteration,
                    "generation_model": LLM_MODEL,
                    "refinement_iteration": iteration,
                }
                new_children.append(child_record)
                accepted_ids.append(child_id)
                print(f"\nIteration {iteration} — accepted child {child_id}")
                for field in ("name", "definition", "parent_id", *review_fields):
                    value = child_record.get(field) if field in {"name", "definition", "parent_id"} else child.get(field)
                    if value is None or value == "":
                        value = "(not provided by the proposal)"
                    print(f"{field}: {json.dumps(value, ensure_ascii=False, indent=2)}")
            progress(
                f"Iteration {iteration}: accepted {len(accepted_ids)} new child/children for {parent_id} "
                f"({NUM_CHILDREN} requested)."
            )
            expanded_parent_ids.add(parent_id)
            parent_results.append({"parent_id": parent_id, "children": accepted_ids, "status": "expanded"})

        if not new_children:
            progress(f"Iteration {iteration}/{N_ITER}: no new child concepts to classify; stopping this search.")
            round_log.append({"iteration": iteration, "parents": parent_results, "children": [], "status": "no_new_children"})
            (RUN_DIR / "iteration_history.json").write_text(json.dumps(round_log, indent=2, ensure_ascii=False))
            print(f"Iteration {iteration}: no new children to evaluate; stopping early.")
            break
        catalog.extend(new_children)
        evaluated_concept_ids.extend(child["concept_id"] for child in new_children)
        current_catalog_version = save_catalog_snapshot(catalog)
        progress(f"Iteration {iteration}/{N_ITER}: saved {len(new_children)} new children; active catalog {current_catalog_version}.")

        # Existing concept labels are reused; only new child definitions require Jev questions.
        scores, signals, current_catalog_version = classify_and_signal(catalog, f"Iteration {iteration} child classification")
        progress(f"Iteration {iteration}/{N_ITER}: beginning forecast evaluation for {len(new_children)} new child concepts.")
        child_ids = [child["concept_id"] for child in new_children]
        children_by_parent = {
            result["parent_id"]: result["children"]
            for result in parent_results if result.get("children")
        }
        _, child_pair_summary, child_population_summary = evaluate_concepts(
            child_ids, f"iteration_{iteration:02d}_children", signals,
            survivor_population=survivor_ids, children_by_parent=children_by_parent,
        )
        if child_population_summary.empty:
            status = "children_not_evaluable"
            progress(f"Iteration {iteration}/{N_ITER}: no child development forecast summaries were available.")
        else:
            all_pair_summaries.append(child_pair_summary)
            all_population_summaries.append(child_population_summary)
            status = "evaluated"
            progress(
                f"Iteration {iteration}/{N_ITER}: evaluation complete — "
                f"{len(child_pair_summary):,} pair/horizon rows and "
                f"{len(child_population_summary):,} population summary rows."
            )
        round_log.append({
            "iteration": iteration,
            "catalog_version": current_catalog_version,
            "parents": parent_results,
            "children": child_ids,
            "status": status,
        })
        (RUN_DIR / "iteration_history.json").write_text(json.dumps(round_log, indent=2, ensure_ascii=False))
        progress(f"Iteration {iteration}/{N_ITER}: round status = {status}; history saved.")
        if status != "evaluated":
            break

    if all_population_summaries:
        population_results = pd.concat(all_population_summaries, ignore_index=True)
        pair_results = pd.concat(all_pair_summaries, ignore_index=True)
        population_results.to_parquet(RUN_DIR / "all_development_population_summaries.parquet", index=False)
        pair_results.to_parquet(RUN_DIR / "all_development_pair_summaries.parquet", index=False)
        final_frontier = pareto_concept_frontier(
            population_results, horizon=0, excluded_concepts=expanded_parent_ids,
        )
        final_frontier.to_csv(RUN_DIR / "final_unexpanded_pareto_frontier.csv", index=False)
        display_columns = [c for c in ["concept_id", "generality", "specialist_strength", "on_pareto_front"] if c in final_frontier]
        progress("Final Pareto frontier (with expansion status):")
        display(final_frontier[display_columns].head(FRONTIER_DISPLAY_ROWS))
    progress(f"Automated development cycle complete. Catalog {current_catalog_version}; outputs: {RUN_DIR}")


## After the automated rounds

Inspect `iteration_history.json`, per-round proposals and forecasts, and the final Pareto table. Confirm the children’s definitions and hierarchy, then use notebook 10e to evaluate the frozen catalog on confirmation pairs. Update its expected catalog hash to the final catalog version before running it. Do not use confirmation results to revise the vocabulary.